第11回講義
========

テキストデータの処理(テキストマイニング)
------------------------------

英語などの西洋のテキストデータでは単語間がスペースで区切られているため、単語による文章の分割は容易です。一方、日本語などではテキスト中に含まれる単語を抜き出し処理する際に、単語や動詞とその送り仮名といいた区切りが不明確なため、**形態要素解析**と呼ばれる前処理が必要となります。

### janomeモジュール

Pythonで形態要素解析を行うモジュールで有名なのは`MeCab`ですが、codespace上で動作させるには、(実行環境に依存する)辞書のインストールなどが面倒なので、代わりに`janome`というモジュールを使います。まずは`janome`をインストールします。

In [ ]:
!pip install janome

<mark>練習1</mark> 青空文庫から引用した太宰治著「走れメロス」のテキストデータ`hashire_merosu.txt`を`janome`を用いて形態要素解析を行いなさい。

In [ ]:
from janome.tokenizer import Tokenizer

f = open("hashire_merosu.txt", "r")

t = Tokenizer()
parsed = t.tokenize(f.read())

f.close()

for token in parsed:
    print(token)

`t`は`Tokenizer()`によって生成されたオブジェクトで、`tokenize`メソッドによって形態要素解析を行います。

`tokenize`メソッドによって解析された結果(トークン)には、表層形(surface)と品詞(part_of_speech)が含まれます。表層形は文章の中で使われているそのままの単語で、品詞にはその単語に関する説明や読みなどが収納されます。

<mark>練習2</mark> 形態要素解析行なったトークンから、品詞が名詞の者だけを抜き出し、リスト化しなさい。

In [ ]:

from janome.tokenizer import Tokenizer

f = open("hashire_merosu.txt", "r")

t = Tokenizer()

parsed = t.tokenize(f.read())

f.close()

list = []

for token in parsed:
    if token.part_of_speech.split(',')[0] == '名詞':
        list.append(token.surface)
        
print(list)

<mark>練習3</mark> `collections`モジュールの`Counter`メソッドを用いて、文章中の名詞の出現頻度について集計しなさい。 

In [ ]:
from janome.tokenizer import Tokenizer
import collections

f = open("hashire_merosu.txt", "r")

t = Tokenizer()

parsed = t.tokenize(f.read())

f.close()

list = []

for token in parsed:
    if token.part_of_speech.split(',')[0] == '名詞':
        list.append(token.surface)

word_count = collections.Counter(list)

print(word_count)

`Counter`クラスは辞書形の拡張となっていて、`most_common()`メソッドを用いると、集計数の多い順に上位の部分だけを取り出すことができる。

<mark>練習4</mark> 名詞の出現頻度から上位13個までを取り出し、棒グラフとして出現頻度を表しなさい。

In [ ]:
!pip install matplotlib_fontja

In [ ]:
from janome.tokenizer import Tokenizer
import collections
import pandas as pd
import matplotlib_fontja

f = open("hashire_merosu.txt", "r")

t = Tokenizer()
parsed = t.tokenize(f.read())

f.close()

list = []

for token in parsed:
    if token.part_of_speech.split(',')[0] == '名詞':
        list.append(token.surface)

word_count = collections.Counter(list)

df = pd.DataFrame(word_count.most_common(13), columns=['word', 'count'])

df.plot(kind='bar', x='word', y='count', title='名詞の出現頻度', figsize=(10, 5), legend=False)

### WordCloudモジュール

WordCloudモジュールは、単語に出現頻度などの値を組み合わせたデータから、その値に応じて大きさを変化させながら表示させるモジュールです。文字の大きさと値が対応するため、視覚的に出現頻度などを把握することができます。

<mark>練習5</mark> `janome`を用いて形態要素解析を行なったデータをWordCloudを用いて表示させなさい。

WordCloudのインストール

In [ ]:
!pip install wordcloud

In [ ]:
from janome.tokenizer import Tokenizer
import collections
from wordcloud import WordCloud
import matplotlib.pyplot as plt

f = open("hashire_merosu.txt", "r")

t = Tokenizer()
parsed = t.tokenize(f.read())

f.close()

words = ""

# 名詞を抽出し、空白で区切った文字列を作成
for token in parsed:
    if token.part_of_speech.split(',')[0] == '名詞':
        words = words + token.surface + " "

wc = WordCloud(
    background_color='white',
    height=600,
    width=800,
    max_words=15,
    stopwords=['の', 'よう'],
    font_path='ipag.ttf' # 日本語フォントのパスを指定
)

wc.generate(words)

plt.figure(figsize=(8, 8))
plt.imshow(wc, interpolation='bilinear')

地理空間データの可視化
----------------

地球上(地図上)の空間に定義された各種データを地理空間データといいます。地理空間データを扱うシステムを**地理情報システム**(GIS: Geographic Information System)といい、PythonでもGISのための各種モジュールが開発されています。

ただ、地理空間データ扱う方法は様々で、標準となる方法が定まっていないため、入手したデータ形式ごとにモジュールやデータ加工の方法を考える必要があります。

この授業では、Python上のGISである`geopandas`というモジュールを用いて、地理空間データを扱う方法を学びます。

以下のコマンドを実行して、まずは`geopandas`モジュールをインストールします。

In [ ]:
!pip install geopandas

`geopandas`はこれまで学んだ`pandas`をGISとして拡張したもので、使い方は`pandas`とほぼ同じです。以下の例では、`gpd`という名前で`geopandas`モジュールを読み込み、`geojson`形式の地理空間データをデータフレームとしてファイルから読み込みます。

<mark>練習6</mark> `land_price.geojson`ファイルからデータを読み込み、geopandasのデータフレームを作成しなさい。

In [ ]:
import geopandas as gpd

gdf = gpd.read_file('land_price.geojson')

`geojson`形式とは、元々はJavaScript用に開発されたJSON(JavaScript Object Notation)形式をGIS用データとして定義を標準化したものです。汎用性が高く、扱いやすいので最近は広まりつつありますが、ファイルサイズが大きくなるという欠点があります。

`geopandas`のデータフレームは、`pandas`のデータフレームと基本的には同じもので、GISで用いられる緯度・経度の情報といった地理データのための特定の列が含まれています。(データフレームオブジェクト内のメソッドがGIS用に拡張されている。)

<mark>練習7</mark> 読み込んだデータフレームの中身を確認しなさい。

In [ ]:
print(gdf.head())

今回用いる地価公示価格のGISデータは、国土交通省の[国土数値情報ダウンロードサイト](https://nlftp.mlit.go.jp/ksj/)から入手し、必要な列だけを抜き出したものです。オリジナルの[データ](https://nlftp.mlit.go.jp/ksj/gml/datalist/KsjTmplt-L01-2025.html)には他のデータ形式やデータも多く含まれていてファイルサイズは巨大になっています。(ダウンロードする際には注意してください。)

地理空間データには大きく分けてビットマップ画像データとして収納されている**ラスタ形式**と座標やポリゴンデータとして収納されている**ベクター形式**があります。`geojson`はベクター形式となっています。

ベクター形式の主な地理空間データ形式には以下のものがあります。
|フォーマット|提唱者|ベースとなるデータ形式|編集で使われるソフトウェア|
|---------|----|-----------------|-------------------|
|シェープファイル|Esri社|オリジナル|ArcGISを初めとするGISソフト|
|GeoJSON|Geographic JSON working group|JSON|テキストエディタ|
|KML|Keyhole社(現Google Earth)|XML|Google Maps, Google Earth|

`geopandas`データフレームが作成できたら、組み込みのメソッドを使って、データのプロットを行うことができます。

<mark>練習8</mark> `geopandas`の`plot`メソッドを用いて、各地点の地価公示価格を色分けして表示しなさい。

In [ ]:
gdf.plot(column='land_price', legend=True, figsize=(10, 10))

このプロットでは、データに含まれる各地点の地価公示価格をその価格に応じて色分けして表示してイアますが、ほとんど同じ色で埋もれてしまっています。一方で、日本の多くの地点が登録されているので、地点のプロットが日本列島の形になっていることがわかります。

<mark>練習9</mark> 地価公示価格データから、500,000円以上1,000,000円未満の地点を抽出してプロットしなさい。

In [ ]:
higher_price = gdf[(gdf['land_price'] >= 500000) & (gdf['land_price'] < 1000000)]

higher_price.plot(column='land_price', legend=True, figsize=(10, 10))

このプロットにより、地価が高い場所は大都市に集中していることがおおよそわかりますが、日本地図上のどの場所かがわかりにくくなってしまいました。

`prefectures.geojson`に各都道府県の県境をポリゴンデータとして収納したGeoJSONファイルがあります。

<mark>練習10</mark> `prefectures.geojson`ファイルを読み込み、日本地図としてプロットしなさい。

In [ ]:
japan = gpd.read_file('prefectures.geojson')
japan.plot(color='lightyellow', edgecolor='gray')


<mark>練習11</mark> `prefectures.geojson`を使って描いた日本地図と、例題4の抽出したデータと重ねて表示させなさい。

In [ ]:
import matplotlib.pyplot as plt

# matplotlibでプロット枠を作成
fig, ax = plt.subplots(1, 1, figsize=(10, 10))

# 地図をプロット
japan = gpd.read_file('prefectures.geojson')
japan.plot(ax=ax, color='lightyellow', edgecolor='gray')

# 地価データを抽出してプロット
higher_price = gdf[(gdf['land_price'] >= 500000) & (gdf['land_price'] < 1000000)]

higher_price.plot(ax=ax, column='land_price', legend=True, figsize=(10, 10))

各都道府県ごとのデータにまとめて、`prefectures.geojson`のデータに追加することで、ポイントではなく、色分け地図として表示させることもできる。

<mark>練習12</mark> 地価公示価格のデータフレームから、各都道府県ごとの地価の**平均値の対数**を集計し、日本地図上に色分けして表示させなさい。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd

gdf = gpd.read_file('land_price.geojson')

# 47都道府県の名前をリストにする
prefs = ['北海道', '青森県', '岩手県', '宮城県', '秋田県',
         '山形県', '福島県', '茨城県', '栃木県', '群馬県',
         '埼玉県', '千葉県', '東京都', '神奈川県', '新潟県',
         '富山県', '石川県', '福井県', '山梨県', '長野県',
         '岐阜県', '静岡県', '愛知県', '三重県', '滋賀県',
         '京都府', '大阪府', '兵庫県', '奈良県', '和歌山県',
         '鳥取県', '島根県', '岡山県', '広島県', '山口県',
         '徳島県', '香川県', '愛媛県', '高知県', '福岡県',
         '佐賀県', '長崎県', '熊本県', '大分県', '宮崎県',
         '鹿児島県', '沖縄県']

# 住所に各都道府県名が含まれるものだけを抽出し、平均値をリストにする
mean_prices = []

for pref in prefs:
    area = gdf[gdf['address'].str.contains(pref)]
    mean_prices.append(area['land_price'].mean())

# 各都道府県の平均地価の対数を持つGeoDataFrameを作成
japan = gpd.read_file('prefectures.geojson')
japan['mean_price_log'] = np.log(mean_prices)

# 平均地価を持つGeoDataFrameをプロット
japan.plot(column='mean_price_log', legend=True, figsize=(10, 10))
